# NB15 탈중앙 거래소와 AMM — 상수곱 풀, 슬리피지, 비영구 손실

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB15-1: 상수곱 풀 — 스왑(수수료 포함), 유동성 추가·제거, 슬리피지 곡선 (공부용)
import math

class ConstantProductPool:
    """x·y = k 풀. fee 는 넣는 양에 매기는 비율(기본 0.003 = 30 bp). 지분은 처음 sqrt(x·y), 이후 비례."""
    def __init__(self, x: float, y: float, fee: float = 0.003):
        self.x, self.y, self.fee = float(x), float(y), fee
        self.shares = {"LP0": math.sqrt(x * y)}            # 첫 공급자의 지분 = 기하평균 (v2 식 (13))
    @property
    def k(self): return self.x * self.y
    @property
    def price_x_in_y(self): return self.y / self.x           # 한계가격: X 하나의 Y 가격 (v2 식 (1))
    def quote_x_for_y(self, dx: float) -> float:              # X 를 dx 넣으면 Y 를 얼마 받나 (수수료 뗀 뒤 곱을 지킴)
        eff = dx * (1 - self.fee)
        return self.y * eff / (self.x + eff)
    def swap_x_for_y(self, dx: float) -> float:
        dy = self.quote_x_for_y(dx)
        self.x += dx; self.y -= dy                             # 수수료 몫은 풀에 남아 k 가 자란다
        return dy
    def swap_y_for_x(self, dy: float) -> float:
        eff = dy * (1 - self.fee)
        dx = self.x * eff / (self.y + eff)
        self.y += dy; self.x -= dx
        return dx
    def add_liquidity(self, who: str, dx: float) -> float:     # 현재 비율대로 X dx 와 Y dx·(y/x) 를 넣는다 (v2 식 (12))
        dy = dx * self.y / self.x
        minted = dx / self.x * sum(self.shares.values())
        self.x += dx; self.y += dy
        self.shares[who] = self.shares.get(who, 0) + minted
        return minted
    def remove_liquidity(self, who: str):                      # 지분 비율만큼 두 토큰을 그때의 비율로 돌려준다
        s_total = sum(self.shares.values())
        frac = self.shares.pop(who) / s_total
        out_x, out_y = self.x * frac, self.y * frac
        self.x -= out_x; self.y -= out_y
        return out_x, out_y

pool = ConstantProductPool(1000, 1000)
print(f"처음: x={pool.x:.1f} y={pool.y:.1f} k={pool.k:,.0f} 한계가격(X 하나의 Y 가격)={pool.price_x_in_y:.4f}")
print(f"{'주문 Δx':>8} {'받는 Δy':>10} {'실행가격 Δy/Δx':>14} {'슬리피지':>9} {'거래 뒤 한계가격':>14}")
for dx in [1, 10, 100, 500, 1000]:
    p = ConstantProductPool(1000, 1000)
    before = p.price_x_in_y
    dy = p.swap_x_for_y(dx)
    exec_price = dy / dx
    print(f"{dx:>8} {dy:>10.3f} {exec_price:>14.4f} {1 - exec_price / before:>9.2%} {p.price_x_in_y:>14.4f}")

# 수수료는 어디로 갔나 — 100 을 넣은 뒤 k 를 본다
p = ConstantProductPool(1000, 1000); dy = p.swap_x_for_y(100)
print(f"\n100 을 넣은 뒤: 받은 Δy={dy:.3f} (수수료 없을 때 90.909), 풀 x={p.x:.1f} y={p.y:.3f}, k={p.k:,.1f} (거래 전 1,000,000 — 수수료 0.3 X 가 풀에 남아 k 가 자랐다)")

처음: x=1000.0 y=1000.0 k=1,000,000 한계가격(X 하나의 Y 가격)=1.0000
   주문 Δx      받는 Δy     실행가격 Δy/Δx      슬리피지      거래 뒤 한계가격
       1      0.996         0.9960     0.40%         0.9980
      10      9.872         0.9872     1.28%         0.9803
     100     90.661         0.9066     9.34%         0.8267
     500    332.666         0.6653    33.47%         0.4449
    1000    499.249         0.4992    50.08%         0.2504

100 을 넣은 뒤: 받은 Δy=90.661 (수수료 없을 때 90.909), 풀 x=1100.0 y=909.339, k=1,000,272.8 (거래 전 1,000,000 — 수수료 0.3 X 가 풀에 남아 k 가 자랐다)


In [ ]:
# NB15-2: 저울의 양쪽 — 가격 배율 r 에 따른 비영구 손실, 그리고 수수료로 자란 sqrt(k) 가 그것을 얼마나 메우는가 (공부용)
import math

def impermanent_loss(r: float) -> float:                     # 이 책의 유도: V_pool / V_hold − 1 = 2√r/(1+r) − 1
    return 2 * math.sqrt(r) / (1 + r) - 1

print("가격 배율 r 과 비영구 손실(수수료 0, 차익거래로 비율 완전 조정 가정):")
print(f"{'r':>6} {'보유 가치(Y 단위)':>16} {'풀 가치(Y 단위)':>16} {'비영구 손실':>10}")
x0 = y0 = 1000.0
for r in [0.25, 0.5, 0.8, 1.0, 1.25, 2.0, 4.0]:
    hold = x0 * r * (y0 / x0) + y0                              # X 를 새 가격 r·(y/x) 로 평가 + Y
    pool_val = 2 * y0 * math.sqrt(r)
    print(f"{r:>6.2f} {hold:>16.1f} {pool_val:>16.1f} {impermanent_loss(r):>10.2%}")

# 수수료 쪽 저울 — 거래가 왕복해 비율은 제자리로 돌아오지만 k 는 자란다. 거래량(풀 대비 배수)별로 sqrt(k) 성장을 센다.
print("\n수수료 적립: 크기 10 의 스왑을 X→Y, Y→X 번갈아 N 번 (비율은 거의 제자리, k 는 자란다)")
print(f"{'왕복 거래량/풀':>12} {'sqrt(k) 성장(=지분당 수수료 적립)':>30} {'V_pool = V_hold 가 되는 r(가정 하의 식 값)':>30}")
for n_pairs in [10, 100, 1000]:
    p = ConstantProductPool(1000, 1000)
    k0 = p.k
    for _ in range(n_pairs):
        p.swap_x_for_y(10); p.swap_y_for_x(10)
    growth = math.sqrt(p.k / k0) - 1                           # g = √(k2/k1) − 1. v2 식 (4) 의 f = 1 − √k1/√k2 와는 f = g/(1+g) 관계(기준 시점이 다르다)
    # 수수료 성장 g 가 비영구 손실을 정확히 상쇄하는 r: (1+g)·2√r/(1+r) = 1 → 2차방정식의 두 근
    a = 1 + growth
    disc = (2 * a) ** 2 - 4
    s1, s2 = (2 * a - math.sqrt(disc)) / 2, (2 * a + math.sqrt(disc)) / 2   # √r 의 두 근
    print(f"{2 * n_pairs * 10 / 1000:>11.1f}x {growth:>30.4%} {f'{s1**2:.3f} ≤ r ≤ {s2**2:.3f}':>30}")

가격 배율 r 과 비영구 손실(수수료 0, 차익거래로 비율 완전 조정 가정):
     r      보유 가치(Y 단위)       풀 가치(Y 단위)     비영구 손실
  0.25           1250.0           1000.0    -20.00%
  0.50           1500.0           1414.2     -5.72%
  0.80           1800.0           1788.9     -0.62%
  1.00           2000.0           2000.0      0.00%
  1.25           2250.0           2236.1     -0.62%
  2.00           3000.0           2828.4     -5.72%
  4.00           5000.0           4000.0    -20.00%

수수료 적립: 크기 10 의 스왑을 X→Y, Y→X 번갈아 N 번 (비율은 거의 제자리, k 는 자란다)
    왕복 거래량/풀        sqrt(k) 성장(=지분당 수수료 적립) V_pool = V_hold 가 되는 r(가정 하의 식 값)
        0.2x                        0.0299%              0.952 ≤ r ≤ 1.050
        2.0x                        0.2985%              0.857 ≤ r ≤ 1.167
       20.0x                        2.9853%              0.614 ≤ r ≤ 1.628
